# E3 — Kernel Ridge RBF

[Artículo](https://doi.org/10.3390/app12094252). Experimento solicitado de la búsqueda Scopus.

Se usa `KernelRidge(kernel='rbf')` de scikit-learn:
la arquitectura KRR del artículo, con solución exacta, **sin reproducir el
optimizador Truncated Newton**. No llamamos TN-KRR a este experimento.

Preparación canónica SVM con log1p en tiempos, escalado y one-hot, aprendida
dentro del fold. Como KernelRidge no incorpora intercepto, el objetivo se
centra dentro de cada fit mediante TransformedTargetRegressor y se devuelve
a la escala original antes de medir. Grilla35: alpha.01/.1/1/10/100 ×
gamma.001/.003/.01/.03/.1/.3/1. Ridge lineal previo no cubría este modelo.

## Protocolo compartido

Misma fuente real `Data/Datos_Brutos.csv`, reglas de negocio y15 predictores.
Objetivo `nota_primer_parcial`; nota_final, pidm, periodo e índice de fila no
son entradas. Sin aumentación ni eliminación de outliers válidos.
Train1309 filas/1293 estudiantes; test202530:188 estudiantes, sin solapamiento.

Cinco folds externos canónicos por estudiante, cinco internos en cada búsqueda
y cinco internos para elegir parámetros finales. RMSE medio interno selecciona
la configuración. Cada preprocesador y modelo se ajusta solo en su fit.
La selección final se guarda antes de predecir test. Mismas funciones del tablero:
RMSE, MAE, MAPE(%) y R²; métricas sin ponderación, escala original.
Media/SD entre folds, OOF reunido y test temporal se informan por separado.

Se guardan grillas, auditorías, hashes, OOF y predicciones test. El código puede
reanudar cálculos de igual procedencia; si cambia la procedencia exige una
carpeta nueva. Pesos/estimadores quedan locales en modelos_preentrenados/scopus.
No se reemplazan los modelos del dashboard ni se reescriben informes anteriores.

In [1]:
from pathlib import Path
import sys
BOOK=Path.cwd() if (Path.cwd()/'entregable3.py').exists() else Path.cwd()/'Precalculo'
sys.path.insert(0,str(BOOK.resolve()))
import pandas as pd
from IPython.display import display
from evaluar_scopus_e3 import run,compare,ROOT

report=run('kernel_ridge')
display(pd.DataFrame(report['cv']).round(6))
print('Parámetros finales:',report['best_params'])

,metrica,media_cv,sd_cv,OOF_agrupado
0,RMSE,0.801325,0.038516,0.802245
1,MAE,0.629252,0.034385,0.629247
2,MAPE (%),23.614417,1.932755,23.613739
3,R²,0.482712,0.051489,0.484725


Parámetros finales: {'alpha': 1.0, 'gamma': 0.03}


## Test temporal y auditoría

El test no selecciona hiperparámetros.

In [2]:
display(pd.DataFrame([report['test']]).round(6))
display(pd.DataFrame(report['fold_audit']))

,RMSE,MAE,MAPE (%),R²
0,0.904881,0.727607,28.32454,0.456316


,fold,best_params,inner_rmse,fit_rows,validation_rows,shared_students,seconds
0,1,"{'alpha': 1.0, 'gamma': 0.03}",0.826398,1047,262,0,19.424431
1,2,"{'alpha': 0.1, 'gamma': 0.01}",0.795523,1047,262,0,17.315235
2,3,"{'alpha': 1.0, 'gamma': 0.03}",0.790714,1047,262,0,16.174648
3,4,"{'alpha': 1.0, 'gamma': 0.03}",0.802883,1047,262,0,15.756796
4,5,"{'alpha': 1.0, 'gamma': 0.03}",0.803904,1048,261,0,15.652505


## Comparación conjunta

Tras completar los cuatro modelos, consultar precalculo_scopus_e3.ipynb y resultados/e3/scopus/comparacion.json.